# 26. Explainable Retrieval

Show *why* each passage was retrieved and cite sources in the answer.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Users trust answers they can verify. **Explainable retrieval** shows *why* each passage was retrieved (relevance explanation) and ties every claim in the answer to a numbered source. That helps users, helps you debug retrieval, and makes hallucinations visible.

## How it works

1. Retrieve top-k passages
2. Ask the LLM to explain each passage's relevance in one sentence
3. Number the passages in the prompt
4. Require inline `[n]` citations in the answer
5. Validate that cited numbers exist and map back to sources

```text
 question --> top-3 --> [explain each] --> "why retrieved"
                    --> numbered context --> LLM --> "Claim A [1]. Claim B [3]."
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 15:52:00.392000 14992 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
def explain(query, chunk):
    return llm(f"In one sentence, explain how this passage relates to the question.\n\nQuestion: {query}\n\nPassage: {chunk}")

query = "How do I stop a model from making things up?"
hits = store.search(query, 3)
for n, h in enumerate(hits, 1):
    print(f"[{n}] similarity={h['score']:.3f}")
    print("    why:", explain(query, h["text"]))
    print("    text:", h["text"][:120], "...\n")

[1] similarity=0.274


    why: The passage relates to the question by mentioning "filtered retrieved text" as a mitigation strategy to prevent indirect prompt injection and ensure that the model does not make things up by limiting what tools it may call, implying that delaying or filtering input can help prevent fabrication.
    text: delay.

## Security and Privacy

Retrieved text is untrusted input. A malicious document can contain instructions that t ...

[2] similarity=0.259


    why: This passage does not directly relate to stopping a model from making things up, but rather discusses the security aspect of running models locally and how it can help protect sensitive information by keeping it on the user's own machine.
    text: user or group identifiers. Running models locally with a tool such as Ollama keeps sensitive documents on the user's own ...

[3] similarity=0.249


    why: This passage relates to the question by providing techniques for grounding a model's responses in reality, such as requiring citations and running verification passes, to prevent the model from making things up or generating false information.
    text: them, because the model can ignore the context or over-generalise from it. Grounding techniques include instructing the  ...



Ask for inline `[n]` citations so every claim can be traced to a source chunk.

In [3]:
numbered = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
print(llm(
    f"Context:\n{numbered}\n\nQuestion: {query}\n\nAnswer using only the context. After each claim add the source number like [1].",
    system=RAG_SYSTEM,
))

According to the context, you can prevent a model from making things up by instructing it to answer only from the context, requiring citations for each claim, and running a second verification pass that checks the answer against the retrieved text. This approach is referred to as grounding techniques.

[1] delay.


## Programmatically verify the citations
A citation like `[7]` when only 3 sources exist is a red flag.

In [4]:
import re
q = "How do reranking and hybrid search help?"
hits = store.search(q, 4)
numbered = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
response = llm(f"Context:\n{numbered}\n\nQuestion: {q}\n\nAnswer using only the context. After each claim add the source number like [1].", system=RAG_SYSTEM)
cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", response)})
print(response, "\n")
print("cited sources:", cited, "| valid:", all(1 <= n <= len(hits) for n in cited), "| unused sources:", [n for n in range(1, len(hits) + 1) if n not in cited])

Reranking and hybrid search help by improving recall in production, as stated in [1]. Reranking uses a cross-encoder to read the query and candidate passage together and outputs a relevance score, which is far more accurate than comparing two independent vectors. This process usually gives a large boost in precision for a modest latency cost, as mentioned in [3]. 

cited sources: [1, 3] | valid: True | unused sources: [2, 4]


## Show the cited text next to each claim

In [5]:
for n in cited:
    if 1 <= n <= len(hits):
        print(f"[{n}] score={hits[n - 1]['score']:.3f}  {hits[n - 1]['text'][:200]}\n")

[1] score=0.739  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted score or with Reciprocal Rank Fusion, which adds 1 divided by (60 plus rank) for every list in which 

[3] score=0.452  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. Because cross-encoders are slow, they are applied only to the top 20 to 50 candidates, and the best 



## Faithfulness spot-check
For each cited claim, does the cited passage really support it?

In [6]:
sentences = [s for s in split_sentences(response) if re.search(r"\[\d+\]", s)]
for s in sentences:
    n = int(re.search(r"\[(\d+)\]", s).group(1))
    if 1 <= n <= len(hits):
        ok = llm_yes(f"Passage: {hits[n - 1]['text']}\n\nClaim: {s}\nDoes the passage support the claim?")
        print("SUPPORTED " if ok else "UNSUPPORTED", "|", s[:110])

SUPPORTED  | Reranking and hybrid search help by improving recall in production, as stated in [1].


SUPPORTED  | This process usually gives a large boost in precision for a modest latency cost, as mentioned in [3].


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Builds user trust | One extra LLM call per source for explanations |
| Makes retrieval debuggable | Small models sometimes cite wrongly |
| Citations can be machine-verified | Longer outputs |

## When to use it

Any user-facing assistant, especially in regulated or high-trust domains.

## Experiments to try

- Highlight the exact supporting sentence instead of the whole chunk.
- Reject answers whose citations fail verification.
- Show similarity scores as a confidence bar.

## Key takeaways

- Number your sources.
- Verify citations in code.
- Explanations double as debugging output.

---
**Next:** [27. Corrective RAG (CRAG)](../27_crag/crag.ipynb)